In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
# os.environ["OPEN_API_KEY"]=os.getenv("OPEN_API_KEY")
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")

### Summarization Middleware

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)



In [7]:
### Run with thread id

config={"configurable":{"thread_id":"test-1"}}

In [8]:
#  Alternative test data
questions=[
    "what is 2+2",
    "what is 10*5",
    "what is the highest rated imdb movie",
    "which is artificial interllignece",
    "what is 4*54",
    "what is a leap year",
    "which is best university to do masters in coumpter science"
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages:{response}")
    print(f"messages:{len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='f3c8b19d-a3c8-4147-92db-668f099d9a5d'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'User asks "what is 2+2". Simple. Provide answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 77, 'total_tokens': 115, 'completion_time': 0.03918602, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.003693451, 'prompt_tokens_details': None, 'queue_time': 0.310093757, 'total_time': 0.042879471}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0946f-6deb-7721-8c70-77ac032079d4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 38, 'total_tokens': 115, 'output_token_details': {'reasoning': 19}})]}
messages:2
Mes

### Token size

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage


@tool
def search_hotels(city:str)->str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa , pool, gym
    2. City Inn - 4 star, $180/night, buisness center
    3. Budget Stay - 3 start,$75/night, free wifi"""


agent=create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)


config={"configurable":{"thread_id":"test-1"}}


def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars//6

In [16]:
cities=["Paris","London","Tokyo", "New York", "Dubai","Singapore"]

for city in cities:
    response=agent.invoke({
        "messages":[HumanMessage(content=f"Find hotels in {city}")]
    },
    config=config
    )
    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens}, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~350, 7 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT  \nUser wants to find hotels in Singapore.\n\n## SUMMARY  \n- Previously invoked `search_hotels` with `city: "Dubai"`.  \n- Returned three options:  \n  1. **Grand Hotel** – 5\u202f★, $350/night, spa, pool, gym.  \n  2. **City Inn** – 4\u202f★, $180/night, business center.  \n  3. **Budget Stay** – 3\u202f★, $75/night, free Wi‑Fi.  \n- Assistant displayed these results in a table and offered next actions: refine the search (price range, star rating, amenities, location), request more details on a specific hotel, compare rates/check availability for a date range, or explore nearby attractions/transportation.  \n- No artifacts were created.  \n- User now requests hotels in Singapore.\n\n## ARTIFACTS  \nNone.\n\n## NEXT STEPS  \n- Invoke `search_hotels` with `city: "Singapore"`.  \n- Offer refinement options (price range, star rating, amenities, location, date range).  \n- Aw

### Human in the Loop MiddleWare

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its ID."""
    return f"Email content for the Id: {email_id}"
def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject {subject}"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
    
)




In [23]:
config={"configurable":{"thread_id":"test-1"}}

# Step 1: Request
result=agent.invoke(
    {"messages":[HumanMessage(content="Sent email to john@test.com with the subject 'Hello' and body 'how are you ?' ")]},
    config=config
)

In [24]:
result

{'messages': [HumanMessage(content="Sent email to john@test.com with the subject 'Hello' and body 'how are you ?' ", additional_kwargs={}, response_metadata={}, id='b65e8df4-cd00-484f-86f7-64f440b92414'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user says "Sent email to john@test.com with the subject \'Hello\' and body \'how are you ?\'". They likely want the assistant to send an email. We should use the send_email_tool. Provide recipient john@test.com, subject Hello, body "how are you ?". Then we should respond with a confirmation. Use the function.', 'tool_calls': [{'id': 'fc_1a9cf1c9-0bf0-473c-97e8-549d53b91c9b', 'function': {'arguments': '{"body":"how are you ?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 177, 'total_tokens': 286, 'completion_time': 0.11932484, 'completion_tokens_details': {'reasoning_tokens': 72}, 'prompt

In [25]:
# Step 2: Approve
from langgraph.types import Command

if "__interrupt__" in result:
    print("|| Paused! Approving...")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result:{result['messages'][-1].content}")

|| Paused! Approving...
Result:✅ Email sent to john@test.com with subject “Hello” and body “how are you ?”.  
Let me know if you’d like to do anything else!
